ChromaDB stores text documents with embeddings. But our data is in rows — numbers and categories. So we need to convert each row into a meaningful text sentence first.

In [ ]:
# imports
import chromadb
from sentence_transformers import SentenceTransformer
from sentence_transformers import util
import pandas as pd
from dotenv import load_dotenv
from anthropic import Anthropic
import os

In [2]:
df_foodprice_c = pd.read_csv("../data/food_prices_cleaned.csv")
df_poverty_c = pd.read_csv("../data/poverty_cleaned.csv")

print(df_foodprice_c.columns)
print()
print(df_poverty_c.columns)

Index(['date', 'admin1', 'admin2', 'market', 'latitude', 'longitude',
       'category', 'commodity', 'commodity_id', 'unit', 'pricetype', 'price',
       'usdprice'],
      dtype='str')

Index(['provider_admin1_name', 'mpi', 'headcount_ratio',
       'intensity_of_deprivation', 'vulnerable_to_poverty',
       'in_severe_poverty', 'reference_period_start', 'reference_period_end'],
      dtype='str')


In [4]:
# function
def food__row_to_text(row):
    return f"In {row['admin1']}, the {row['pricetype']} price of {row['commodity']} was {row['price']} INR per {row['unit']} in {row['date']}"


In [11]:
# function test
row1 = df_foodprice_c.iloc[0]
print(food__row_to_text(row1))

In Delhi, the Retail price of Rice was 8.0 INR per KG in 1994-01-15


In [12]:
# making a new column and savings senteces formed in that coliumns for every row
df_foodprice_c['text'] = df_foodprice_c.apply(food__row_to_text, axis=1)

In [13]:
df_foodprice_c.head(2)

,date,admin1,admin2,market,latitude,longitude,category,commodity,commodity_id,unit,pricetype,price,usdprice,text
0,1994-01-15,Delhi,Delhi,Delhi,28.67,77.22,cereals and tubers,Rice,52,KG,Retail,8.0,0.26,"In Delhi, the Retail price of Rice was 8.0 INR..."
1,1994-01-15,Delhi,Delhi,Delhi,28.67,77.22,cereals and tubers,Wheat,84,KG,Retail,5.0,0.16,"In Delhi, the Retail price of Wheat was 5.0 IN..."


In [18]:
# function for poverty df
def poverty_row_to_text(row):
    return f"In {row['provider_admin1_name']}, {round(row['headcount_ratio'],2)}% of population lives in poverty with an MPI score of {row['mpi']}"

In [19]:
# function test
row2 = df_poverty_c.iloc[0]
print(poverty_row_to_text(row2))

In Andaman & Nicobar Islands, 3.67% of population lives in poverty with an MPI score of 0.0142


In [20]:
# making a new column and savings senteces formed in that columns for every row
df_poverty_c['text'] = df_poverty_c.apply(poverty_row_to_text, axis=1)

In [21]:
df_poverty_c.head(2)

,provider_admin1_name,mpi,headcount_ratio,intensity_of_deprivation,vulnerable_to_poverty,in_severe_poverty,reference_period_start,reference_period_end,text
0,Andaman & Nicobar Islands,0.0142,3.6708,38.6353,10.1618,0.4575,2019-01-01,2021-12-31,"In Andaman & Nicobar Islands, 3.67% of populat..."
1,Andhra Pradesh,0.2356,49.8836,47.2247,18.6133,21.5491,2005-01-01,2006-12-31,"In Andhra Pradesh, 49.88% of population lives ..."


In [22]:
# saving these dfs
df_foodprice_c.to_csv('../data/food_prices_final.csv', index=False)
df_poverty_c.to_csv('../data/poverty_final.csv', index=False)

print("Saved.")

Saved.


In [ ]:
# craeting client and collections and embedding

chroma_client = chromadb.PersistentClient("../vectorstore/food_poverty_db/")
collection_foodprice = chroma_client.get_or_create_collection(name= 'food_prices')
collection_poverty = chroma_client.get_or_create_collection(name='poverty_mpi')

embed_model = SentenceTransformer('all-MiniLM-L6-v2')

batch_size = 1000

# ingestion of food price data into chromadb
for i in range(0, len(df_foodprice_c), batch_size):
    batch = df_foodprice_c.iloc[i: i + batch_size]
    
    documents = batch['text'].tolist()
    ids = [f"food_{j}" for j in range(i,  i + len(batch))]
    metadatas = [
        {
            'admin1': row['admin1'],
            'commodity': row['commodity'],
            'date': str(row['date']),
            'price': float(row['price'])
        }
        for _, row in batch.iterrows()
    ]
    
    collection_foodprice.add(
        documents= documents,
        ids = ids,
        metadatas = metadatas
    )
    print('done')


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done


In [ ]:
# ingestion of poverty data into chromadb

for i in range(0, len(df_poverty_c), batch_size):
    batch = df_poverty_c.iloc[i: i + batch_size]
    
    documents = batch['text'].tolist()
    ids = [f"poverty_{j}" for j in range(i, i + len(batch))]
    metadatas = [
        {
            'admin1': row['provider_admin1_name'], # as we defined states as admin1 in food data, we used same denotion-"admin1"
            'mpi': row['mpi'],
            'population': row['headcount_ratio'],
            'date': str(row['reference_period_start']) # same as admin1
        }
        for _, row in batch.iterrows()
    ]
    
    collection_poverty.add(
        documents=documents,
        
        ids=ids,
        metadatas=metadatas
    )
    print(f"Added batch {i}")

Added batch 0


In [29]:
# verification
print("Food price count: ", collection_foodprice.count())
print("Poverty count: ", collection_poverty.count())

Food price count:  205431
Poverty count:  94
